# Generic GRN inference pipeline figure

This is a method-agnostic pipeline in the same style and palette as the thesis overview figure. The stages, method families, and metrics follow Pratapa et al. (BEELINE). Specifically:
- Method grouping: the algorithm descriptions in their *Methods* section.
- Optional pseudotime step: Slingshot, used only by time-dependent methods.
- AUROC/AUPR computed on the full ranked list; early precision and EPR on the top-$k$ edges (their *Evaluation Pipeline*).

The "Regression / ODE" family label is my grouping: SINCERITIES is Granger-causality ridge regression, while SCODE and GRISLI are linear-ODE methods.

All text is in `TEXT`; colours are in `PALETTE`. Outputs: `grn_pipeline.pdf` / `.svg` / `.png`.

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle

# ------------------------------------------------------------------ config (same palette as the overview figure)
PALETTE = {
    "data": ("#EEF0FA", "#3949AB"), "infer": ("#FFF4E8", "#E07B00"), "eval": ("#E8F5F3", "#00897B"),
    "text": "#1F2430", "muted": "#5A6072", "card": "#FFFFFF", "line": "#7A8095",
}
TEXT = {
    "expr": ("Expression data", "single-cell counts\ngenes × cells"),
    "prep": ("Preprocessing", "normalisation,\ngene filtering"),
    "ptime": ("Pseudotime (optional)", "cell ordering, e.g. Slingshot;\nneeded by time-based methods"),
    "infer": ("GRN inference", [
        ("Tree ensembles", "GENIE3, GRNBoost2"),
        ("Information theory", "PIDC, SCRIBE"),
        ("Correlation", "PPCOR, LEAP"),
        ("Regression / ODE", "SCODE, GRISLI, SINCERITIES"),
    ]),
    "ranked": ("Ranked edge list", "score for every\nregulator → target pair"),
    "topk": ("Predicted network", "top-$k$ or score threshold;\nedges may be directed/signed"),
    "ref": ("Reference network", "simulated ground truth or\ncurated gold standard"),
    "eval": ("Metrics", [
        ("Threshold-free", "AUROC, AUPR"),
        ("Top-$k$", "early precision, EPR"),
    ]),
    "lbl_all": "all edges",
}
FONT = "DejaVu Sans"
rng = np.random.default_rng(7)

# ------------------------------------------------------------------ helpers
def box(ax, x, y, w, h, fc, ec, lw=1.0, r=1.2, z=1, ls="-"):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle=f"round,pad=0,rounding_size={r}",
                                fc=fc, ec=ec, lw=lw, zorder=z, ls=ls))

def arrow(ax, p0, p1, color, lw=1.5, ms=11, conn="arc3", ls="-", z=5):
    ax.add_patch(FancyArrowPatch(p0, p1, arrowstyle="-|>", mutation_scale=ms, color=color, lw=lw,
                                 connectionstyle=conn, linestyle=ls, zorder=z, shrinkA=0, shrinkB=0))

def inset(ax, x, y, w, h):
    a = ax.inset_axes([x, y, w, h], transform=ax.transData)
    a.set_xticks([]); a.set_yticks([])
    for s in a.spines.values(): s.set_visible(False)
    a.patch.set_alpha(0); return a

def card(ax, x, y, w, h, title, sub, accent, border, ls="-", tsize=9.0, ssize=7.2):
    box(ax, x, y, w, h, PALETTE["card"], border, lw=.9, r=1, ls=ls)
    ax.add_patch(Rectangle((x + .9, y + h - .55), w - 1.8, .55, fc=accent, ec="none", zorder=3))
    ax.text(x + w/2, y + h - 1.5, title, ha="center", va="top", fontsize=tsize, fontweight="bold",
            color=PALETTE["text"], zorder=6)
    if sub:
        ax.text(x + w/2, y + h - 4.0, sub, ha="center", va="top", fontsize=ssize, color=PALETTE["muted"],
                linespacing=1.35, zorder=6)

# ---- icons
def icon_matrix(a, c):
    M = rng.gamma(.9, 1, (9, 14)); M[M < .5] = 0; M[1, 2:7] += 2.5; M[6, 7:12] += 2
    cmap = matplotlib.colors.LinearSegmentedColormap.from_list("m", ["#FFFFFF", c])
    a.imshow(M, cmap=cmap, aspect="auto", interpolation="nearest")
    for s in a.spines.values(): s.set_visible(True); s.set_color("#B0B4C0"); s.set_linewidth(.6)

def icon_norm(a, c):
    x = rng.lognormal(0, .9, 400)
    a.hist(np.log1p(x*4), bins=16, color=c, alpha=.85, ec="white", lw=.5)
    a.spines["bottom"].set_visible(True); a.spines["bottom"].set_color("#9095A5"); a.spines["bottom"].set_linewidth(.6)

def icon_traj(a, c):
    t = rng.uniform(0, 1, 160); br = rng.integers(0, 2, 160)
    x = t*2; y = np.where(t > .45, (t - .45)*(1.6*br - .8), 0) + rng.normal(0, .06, 160)
    cmap = matplotlib.colors.LinearSegmentedColormap.from_list("t", ["#C5CAE9", c])
    a.scatter(x, y, c=t, cmap=cmap, s=5, lw=0)
    tt = np.linspace(0, 1, 50)
    for s in (-.8, .8):
        a.plot(tt*2, np.where(tt > .45, (tt - .45)*s, 0), color="#1F2430", lw=.9)
    a.set_xlim(-.1, 2.1); a.set_ylim(-.6, .6)

def icon_table(a, c):
    rows = [("g3", "g7", .94), ("g1", "g4", .88), ("g7", "g2", .71), ("g5", "g1", .52), ("g2", "g6", .31)]
    a.set_xlim(0, 1); a.set_ylim(0, len(rows) + .2)
    for i, (u, v, s) in enumerate(rows):
        yy = len(rows) - i - .5
        a.text(.02, yy, f"{u} → {v}", fontsize=6.6, va="center", color=PALETTE["text"], family="monospace")
        a.add_patch(Rectangle((.52, yy - .28), .46*s, .56, fc=c, alpha=.35 + .6*s, ec="none"))
    a.axhline(len(rows) - 2 - .0, xmin=0, xmax=1, color=c, lw=.9, ls="--")  # top-k cut

def icon_net(a, c, n=7, E=None, col_edges="#8A90A6"):
    th = np.linspace(0, 2*np.pi, n, endpoint=False) + .3
    xy = np.c_[np.cos(th), np.sin(th)]
    E = E or [(0,1),(1,2),(2,3),(0,4),(4,5),(5,6),(6,2),(3,5)]
    for i, j in E:
        a.annotate("", xy[j], xy[i], arrowprops=dict(arrowstyle="-|>", color=col_edges, lw=.8,
                   shrinkA=4.5, shrinkB=4.5, mutation_scale=6))
    a.scatter(xy[:,0], xy[:,1], s=30, color=c, zorder=3, ec="white", lw=.8)
    a.set_xlim(-1.35, 1.35); a.set_ylim(-1.35, 1.35); a.set_aspect("equal")

def icon_roc(a, c):
    t = np.linspace(0, 1, 100)
    a.plot(t, t**.4, color=c, lw=1.6); a.plot([0,1],[0,1], ls="--", color="#6B7080", lw=.8)
    a.set_xlim(0,1); a.set_ylim(0,1.02)
    for s in ("left","bottom"): a.spines[s].set_visible(True); a.spines[s].set_color("#9095A5"); a.spines[s].set_linewidth(.6)

# ------------------------------------------------------------------ figure
def draw(save_stem=None):
    plt.rcParams.update({"font.family": FONT, "mathtext.fontset": "dejavusans",
                         "pdf.fonttype": 42, "svg.fonttype": "none"})
    W, H = 100, 40
    fig = plt.figure(figsize=(12, 12*H/W))
    ax = fig.add_axes([0, 0, 1, 1]); ax.set_xlim(0, W); ax.set_ylim(0, H); ax.axis("off")
    dF, dA = PALETTE["data"]; iF, iA = PALETTE["infer"]; eF, eA = PALETTE["eval"]; L = PALETTE["line"]

    # background bands
    box(ax, 1, 1, 30.5, 38, dF, "none", r=2)
    box(ax, 33.5, 1, 45.5, 38, iF, "none", r=2)
    box(ax, 81, 1, 18, 38, eF, "none", r=2)
    for x, w, t, c in [(1, 30.5, "DATA", dA), (33.5, 45.5, "INFERENCE", iA), (81, 18, "EVALUATION", eA)]:
        ax.text(x + w/2, 37.6, t, ha="center", va="top", fontsize=10.5, fontweight="bold", color=c)

    top_y, ch = 18.5, 16.0          # main row
    # 1 expression
    x1, w1 = 2.6, 13.0
    card(ax, x1, top_y, w1, ch, *TEXT["expr"], dA, "#D5D9EA")
    icon_matrix(inset(ax, x1 + 1.8, top_y + 1.2, w1 - 3.6, 6.2), dA)
    # 2 preprocessing
    x2, w2 = 17.3, 12.6
    card(ax, x2, top_y, w2, ch, *TEXT["prep"], dA, "#D5D9EA")
    icon_norm(inset(ax, x2 + 1.8, top_y + 1.2, w2 - 3.6, 6.0), dA)
    # 2b pseudotime (optional)
    pw, ph, py = 18.0, 13.0, 2.6
    px = x2 + w2/2 - pw/2 - 2.5
    card(ax, px, py, pw, ph, *TEXT["ptime"], dA, "#B7BEDD", ls=(0, (3, 2)), tsize=8.4, ssize=6.8)
    icon_traj(inset(ax, px + 2.5, py + 1.0, pw - 5, 4.4), dA)
    # 3 inference
    x3, w3 = 35.0, 15.6
    t, fams = TEXT["infer"]
    box(ax, x3, top_y - 15.9, w3, ch + 15.9, PALETTE["card"], iA, lw=1.4, r=1)
    ax.text(x3 + w3/2, top_y + ch - 1.5, t, ha="center", va="top", fontsize=9.6, fontweight="bold", color=iA)
    fy = top_y + ch - 5.2
    for name, ex in fams:
        box(ax, x3 + 1.2, fy - 5.6, w3 - 2.4, 5.6, iF, "none", r=.8, z=2)
        ax.text(x3 + w3/2, fy - 1.2, name, ha="center", va="top", fontsize=8.0, fontweight="bold", color=PALETTE["text"], zorder=6)
        ax.text(x3 + w3/2, fy - 3.4, ex, ha="center", va="top", fontsize=6.9, color=PALETTE["muted"], zorder=6)
        fy -= 6.6
    # 4 ranked list
    x4, w4 = 52.4, 11.6
    card(ax, x4, top_y, w4, ch, *TEXT["ranked"], iA, "#EBD6C0", tsize=8.6, ssize=6.9)
    icon_table(inset(ax, x4 + 1.0, top_y + 1.0, w4 - 2.0, 7.0), iA)
    # 5 top-k network
    x5, w5 = 65.8, 12.4
    card(ax, x5, top_y, w5, ch, *TEXT["topk"], iA, "#EBD6C0", tsize=8.4, ssize=6.5)
    icon_net(inset(ax, x5 + 1.6, top_y + .9, w5 - 3.2, 7.0), iA)
    # 6 evaluation
    x6, w6 = 82.4, 15.2
    t, parts = TEXT["eval"]
    box(ax, x6, top_y, w6, ch, PALETTE["card"], eA, lw=1.4, r=1)
    ax.text(x6 + w6/2, top_y + ch - 1.5, t, ha="center", va="top", fontsize=9.6, fontweight="bold", color=eA)
    for k, (name, mets) in enumerate(parts):
        yy = top_y + ch - 5.0 - k*3.9
        ax.text(x6 + 1.4, yy, name, fontsize=7.8, fontweight="bold", color=PALETTE["text"], va="top")
        ax.text(x6 + 1.4, yy - 1.8, mets, fontsize=7.0, color=PALETTE["muted"], va="top")
    icon_roc(inset(ax, x6 + w6 - 5.6, top_y + .9, 4.4, 3.3), eA)
    # 7 reference network
    rw, rh, ry = 15.2, 13.0, 2.6
    card(ax, x6, ry, rw, rh, *TEXT["ref"], eA, "#C9E3DF", tsize=8.6, ssize=6.8)
    icon_net(inset(ax, x6 + rw/2 - 2.6, ry + .7, 5.2, 4.8), eA, E=[(0,1),(1,2),(2,3),(0,4),(4,5),(5,6),(6,2)])

    # ---- flow arrows
    mid = top_y + ch/2
    arrow(ax, (x1 + w1, mid), (x2 - .1, mid), L)
    arrow(ax, (x2 + w2, mid), (x3 - .1, mid), L)
    arrow(ax, (x2 + w2/2 - 2.5, top_y), (x2 + w2/2 - 2.5, py + ph + .1), L, ls=(0, (3, 2)))   # prep -> pseudotime
    arrow(ax, (px + pw, py + ph/2), (x3 - .1, py + ph/2), L, ls=(0, (3, 2)))                  # pseudotime -> inference
    arrow(ax, (x3 + w3, mid), (x4 - .1, mid), L)
    arrow(ax, (x4 + w4, mid), (x5 - .1, mid), L)
    arrow(ax, (x5 + w5, mid - 3.0), (x6 - .1, mid - 3.0), L)
    # ranked list -> evaluation over the top (all edges, threshold-free)
    yb = top_y + ch + 1.3
    ax.plot([x4 + w4/2, x4 + w4/2], [top_y + ch, yb], color=L, lw=1.5, zorder=4)
    ax.plot([x4 + w4/2, x6 + 3.0], [yb, yb], color=L, lw=1.5, zorder=4)
    arrow(ax, (x6 + 3.0, yb), (x6 + 3.0, top_y + ch + .05), L)
    ax.text((x4 + w4/2 + x6 + 3.0)/2, yb + .4, TEXT["lbl_all"], ha="center", va="bottom", fontsize=6.8, color=PALETTE["muted"])
    # reference -> evaluation
    arrow(ax, (x6 + w6/2, ry + rh), (x6 + w6/2, top_y - .1), eA)

    if save_stem:
        for ext in ("pdf", "png", "svg"):
            fig.savefig(f"{save_stem}.{ext}", dpi=300, bbox_inches="tight", pad_inches=0.05)
    return fig

In [ ]:
fig = draw(save_stem="grn_pipeline")
plt.show()

### LaTeX include (suggested)
```latex
\begin{figure}[t]
  \centering
  \includegraphics[width=\textwidth]{figures/grn_pipeline.pdf}
  \caption{A generic pipeline for GRN inference from single-cell expression data.
  Time-dependent methods additionally require a cell ordering (pseudotime).
  Threshold-free metrics are computed on the full ranked edge list, early precision
  on the top-$k$ edges, with $k$ the number of edges in the reference network.
  Method families and metrics follow Pratapa et al.\ (BEELINE).}
  \label{fig:grn-pipeline}
\end{figure}
```